This is the endpoint for experiment 1 - a barebones lakebase model

In [0]:
UC_MODEL_NAME = "cjc.default.adder_model"
ENDPOINT_NAME = "shm_adder_model_endpoint"

In [0]:
from typing import Dict, Any
import pandas as pd
import mlflow

class AdderModel(mlflow.pyfunc.PythonModel):
    """
    Blank pyfunc to test latency
    """
    def __init__(self):
        # Put any initialization logic here (load artifacts, etc.)
        pass

    def load_context(self, context):
        pass

    def predict(self, model_input: pd.DataFrame) -> Dict[str, Any]:
        sum_result = (
          model_input.iloc[:, 0] 
          + model_input.iloc[:, 1] 
          + model_input.iloc[:, 2]
        ).astype(float)

        # Return in Databricks Model Serving tabular format
        return sum_result

In [0]:
model = AdderModel()

df = pd.DataFrame(
    data=[(1., 2., 3.), (4., 5., 6.), (7., 8., 9.)],
    columns=['col1', 'col2', 'col3']
)
display(df)

In [0]:
model = AdderModel()
model.predict(df)

In [0]:
# Ship as a model and serve as a CPU endpoint
with mlflow.start_run():
    model = AdderModel()
    
    model_info = mlflow.pyfunc.log_model(
      artifact_path="model",
      python_model=model,
      registered_model_name=UC_MODEL_NAME,
      input_example=df,
    )

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedModelInput,
    EndpointCoreConfigInput,
    ServedModelInputWorkloadSize,
    ServedModelInputWorkloadType,
)

w = WorkspaceClient()

served_model = ServedModelInput(
    model_name=UC_MODEL_NAME,
    model_version=model_info.registered_model_version,
    workload_type=ServedModelInputWorkloadType.CPU,
    workload_size=ServedModelInputWorkloadSize.SMALL,
    scale_to_zero_enabled=True
)

config = EndpointCoreConfigInput(served_models=[served_model])

In [0]:
try:
    w.serving_endpoints.update_config(
        name=ENDPOINT_NAME,
        served_models=[served_model],
    ).result()
    print(f"✅ Endpoint '{ENDPOINT_NAME}' updated")
except Exception:
    w.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=config,
    ).result()
    print(f"✅ Endpoint '{ENDPOINT_NAME}' created")